### **folder**

In [ ]:
import torch
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler
from diffusers.utils import load_image
from PIL import Image
import cv2
import numpy as np
import os 
import random

In [ ]:
prompt="Photorealistic style transfer. Apply the lighting, color palette, and texture of the [Style Image] to the background of the [Content Image]. Preserve the original foreground subject's form, colors, and details. High-resolution, cinematic lighting, with natural color blending."


prompt="Apply the style of [Style Image] to the [Content Image] with a focus on the background. Style weight: 0.6. Isolate the foreground subject and maintain its original appearance. Generate a photorealistic image with 8K resolution, detailed textures, and soft, natural lighting to minimize color bleed."

In [ ]:
def change(content_image_url,style_image_url,pipe,generator):

    content_image = Image.open(content_image_url)
    style_image = Image.open(style_image_url)

    canny_image = np.array(content_image)
    canny_image = cv2.Canny(canny_image, 100, 200)
    canny_image = canny_image[:, :, None]
    canny_image = np.concatenate([canny_image, canny_image, canny_image], axis=2)
    canny_image = Image.fromarray(canny_image)


    stylized_image = pipe(
        prompt="Apply the style of [Style Image] to the [Content Image] with a focus on the background. Style weight: 0.6. Isolate the foreground subject and maintain its original appearance. Generate a photorealistic image with 8K resolution, detailed textures, and soft, natural lighting to minimize color bleed.",
        negative_prompt="monochrome, lowres, bad anatomy, worst quality, low quality",
        image=canny_image, # ControlNet input
        ip_adapter_image=style_image, # IP-Adapter input
        num_inference_steps=50,
        generator=generator,
    ).images[0]


    return stylized_image

In [11]:
syn_address = "train\\New folder\images"
real_address = "1"
save_address = "data"

In [12]:
_syn_address = os.listdir(syn_address)
_real_address = os.listdir(real_address)

In [ ]:
base_model_path = "runwayml/stable-diffusion-v1-5"
controlnet_path = "lllyasviel/sd-controlnet-canny"

controlnet = ControlNetModel.from_pretrained(controlnet_path, torch_dtype=torch.float16)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    base_model_path,
    controlnet=controlnet,
    torch_dtype=torch.float16
).to("cuda")

pipe.load_ip_adapter("h94/IP-Adapter", subfolder="models", weight_name="ip-adapter_sd15.bin")

generator = torch.Generator().manual_seed(42)

Loading pipeline components...: 100%|██████████| 7/7 [00:01<00:00,  5.62it/s]


In [ ]:
for i in _syn_address[:]:
    # get content image
    content_address = os.path.join(syn_address,i)

    # get style image
    style_name = random.choice(_real_address)
    style_address = os.path.join(real_address,style_name)

    # create new image 
    new_image = change(content_address,style_address,pipe,generator)

    # save image 
    save_name = os.path.join(save_address,""+i)
    new_image.save(save_name)
